In [27]:
import pygbif
import pandas as pd
import os
import itertools
import shapely
import json
from pathlib import Path
from getpass import getpass
import time

# GBIF taxonomy

## Name reading

In [89]:
def read_species_names(inpFile, inpPath="", sep=","):
    """
    A function to read in the names of different species of interest that are stored in a csv file.
    This function provides automatic formatting

    Args:
        inp_file (str): The name for the file containing all the species names
        inp_path (str, optional): Path where the input file is stored. The default is empty and will be read from the working directory
        sep (str, optional): The separator character used in between species names. The standard used is ',', i.e. comma separated values
    Returns:
        species_names (pd.Dataframe): A dataframe containing all the relevant information stored in the GBIF taxonomic backbone
    """
    with open(os.path.join(inpPath, inpFile), "r") as f:
        fileLines = f.readlines()
    #Read lines up until second last character to drop '\n' new line command
    #Split the lines based on the used separator in the file
    species_names = [line[:-2].split(sep) for line in fileLines]
    #Convert list of lists into a single list
    species_names = list(itertools.chain.from_iterable(species_names))
    #Trim whitespace from the names
    species_names = [name.strip() for name in species_names]
    #Remove the empty elements from the list
    species_names = list(filter(None, species_names))
    #Remove duplicates and return list of unique species
    return list(set(species_names))

def extract_keys_dwc(inp_file, inp_path="", sep="\t", encoding="utf-8"):
    """
    Extract the usagekeys from a Darwin Core archive

    Args
        inp_file (str): the file containing the taxonomic information 
        inp_path (str, optional): The path towards the darwin core taxon file. Standard value is the current working directory
        sep (str, optional): Separator used in the inp_file. Standard separator used in the tab value 
        encoding (str, optional): Encoding used within the file. Standard encoding is utf-8
    Returns
        usageKeys (list<str>): A list containing the usageKeys described within the archive. These keys are multidigit strings
    """
    #Read in the DwC file immediately as a DF with a tab separator and utf8 encoding
    dwc_df = pd.read_csv(os.path.join(inp_path, inp_file), sep="\t", encoding="utf8")
    #Extract the usageKey from the taxonID field in the dataframe
    usageKeys = [key.split("/")[-1] for key in dwc_df["taxonID"]]
    return usageKeys


### Example

In [93]:
inp_path = "scripts/prototype/inp/"
inv_filepath = os.path.join(inp_path, "invasiveNames.txt")
species_list = extract_keys_dwc("invasiveNames.txt", inp_path=inp_path)
species_list

['1002621',
 '1003567',
 '10071055',
 '1007534',
 '1008610',
 '1008612',
 '1008955',
 '1010644',
 '10108775',
 '1013526',
 '1014565',
 '1016841',
 '1017419',
 '10269496',
 '1031394',
 '1031400',
 '1031512',
 '1031520',
 '1031524',
 '1031564',
 '1031677',
 '1031680',
 '1031684',
 '1031685',
 '1031737',
 '1031742',
 '1031743',
 '1032262',
 '1032377',
 '10329298',
 '10411852',
 '1043717',
 '1043978',
 '1045323',
 '1047536',
 '10545407',
 '10578411',
 '10629881',
 '10646747',
 '10676000',
 '10701161',
 '10730110',
 '10755213',
 '10773250',
 '10786206',
 '10797854',
 '10800064',
 '10801424',
 '10852031',
 '10857535',
 '10902460',
 '10920460',
 '10933479',
 '10937982',
 '10944522',
 '10948891',
 '10953644',
 '1095946',
 '10966302',
 '10972541',
 '10986555',
 '11007246',
 '11055820',
 '11064584',
 '11104870',
 '11107889',
 '1111797',
 '11136683',
 '11141765',
 '11162940',
 '1119292',
 '11205618',
 '11335341',
 '1133603',
 '1152186',
 '11528335',
 '11794300',
 '11844792',
 '11921804',
 '119488

In [108]:
import time
begin_time = time.time()
[pygbif.species.name_backbone(name) for name in list(df["scientificName"])]
end_time = time.time()
print(end_time-begin_time)

300.0374035835266


In [107]:
from concurrent.futures import ThreadPoolExecutor, as_completed
begin_time = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(ex.map(pygbif.species.name_backbone, list(df["scientificName"])))
end_time = time.time()
print(end_time-begin_time)

38.90769720077515


In [115]:
def lookup_backbone(name):
    return pygbif.species.name_backbone(name)

%timeit [lookup_backbone(name) for name in list(df["scientificName"])[:100]]

7.8 s ± 357 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [116]:
from functools import lru_cache

def lookup_backbone(name_list):
    with ThreadPoolExecutor(max_workers=8) as ex:
        results = list(ex.map(pygbif.species.name_backbone, name_list))
    return results

%timeit lookup_backbone(list(df["scientificName"])[:100])

1.04 s ± 50.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [122]:
@lru_cache(maxsize=None)
def lookup_backbone(name):
    return pygbif.species.name_backbone(name)

begin_time = time.time()
[lookup_backbone(name) for name in list(df["scientificName"])]
end_time = time.time()
print(end_time-begin_time)

297.90611696243286


In [123]:
@lru_cache(maxsize=None)
def lookup_backbone_single(name: str):
    """Cached single-name GBIF backbone lookup."""
    return pygbif.species.name_backbone(name)

def lookup_backbone(name_list, max_workers=8):
    """Parallel backbone lookup with caching."""
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(lookup_backbone_single, name_list))
        
begin_time = time.time()
%timeit lookup_backbone(list(df["scientificName"]))
end_time = time.time()
print(end_time-begin_time)

56.8 ms ± 7.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
38.7481050491333


## Backbone matching

In [161]:
@lru_cache(maxsize=None)
def lookup_backbone_single(name: str):
    """Cached single-name GBIF backbone lookup."""
    return pygbif.species.name_backbone(name)

def lookup_backbone(name_list, max_workers=8):
    """Parallel backbone lookup with caching."""
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(lookup_backbone_single, name_list))
        
def fetch_taxon_info(inp_file, 
                     inp_path="", 
                     out_file="", out_path="", 
                     sep=",", 
                     mismatch_file= "", 
                     keep_higherrank=False,
                     keep_fuzzy = False,
                     max_workers = 8):
    """
    A function that reads in a list of species names and than retrieves all the relevant information from the GBIF taxonomic backbone. 
    The dataframe containing all the taxonomic data is than formatted to store the taxonKey that is used for the most general name within the column 'acceptedUsageKey'.

    Args:
        inp_file (str): The name for the file containing all the species names
        inp_path (str, optional): Path where the input file is stored. The default is empty and will be read from the working directory
        out_file (str): The name for the file to which the taxonomic information will be written to. If nothing is provided than the result will not be saved
        out_path (str, optional): Path where the output file will be stored. The default is empty and will be written to the working directory
        sep (str, optional): Separator used in the input file in between species names
        mismatch_file (str, optional): File where all the mismatched names should be written to together with the note of the name_backbone function
        keep_higherrank (bool, optional): Boolean option that allows the removal of higherrank matches in the GBIF taxonomic backbone
    Returns:
        taxonomic_df, mismatch_df (pd.Dataframe): A pair of dataframes containing all the relevant information stored in the GBIF taxonomic backbone. The first dataframe is the dataframe with valid matches whereas the second one contains all erroneous matches
    """
    #Retrieve names from the file
    species_names = read_species_names(inp_file, inp_path, sep=sep)
    #Retrieve information from the GBIF taxonomic backbone
    taxonomic_info = lookup_backbone(species_names, max_workers=max_workers)
    #Convert the list of dictionaries to dataframe
    taxonomic_df = pd.DataFrame(taxonomic_info)
    taxonomic_df["lookupNames"] = species_names
    #Extract rows that are None matches and if enabled higherrank matches
    mismatchTypes = ["NONE",
                     *(["FUZZY"] if not keep_fuzzy else []),
                     *(["HIGHERRANK"] if not keep_higherrank else [])]
    mismatch_indices = taxonomic_df["matchType"].isin(mismatchTypes)
    if set(mismatch_indices)=={False, True}:
        print(f"Mismatches encountered of type {set(taxonomic_df['matchType'])}"
              f"{''.join(f'\nCount {mt}: {sum(taxonomic_df['matchType'] == mt)}' for mt in set(taxonomic_df['matchType']))}")
    mismatch_df = taxonomic_df[mismatch_indices]
    #Invert mismatch indices to extract correct matches
    taxonomic_df = taxonomic_df[~mismatch_indices]
    #Assert that usageKeys are cast as integers
    taxonomic_df["acceptedUsageKey"] = taxonomic_df["acceptedUsageKey"].fillna(taxonomic_df["usageKey"])
    # Assure that the keys are stored and represented as integers
    taxonomic_df["acceptedUsageKey"] = taxonomic_df["acceptedUsageKey"].astype(int)
    #If an out_file is specified than the taxonomic info will be written to a file of said name
    if out_file != "":
        taxonomic_df.to_csv(os.path.join(out_path, out_file), index=False)
    if mismatch_file != "":
        mismatch_df[["matchType", "note", "scientificName", "lookupNames"]].to_csv(os.path.join(out_path, mismatch_file), index=False)
    return taxonomic_df, mismatch_df

### Example

In [179]:
tdf1, mmdf1 = fetch_taxon_info("prototypeNames.csv",
                               inp_path = "scripts/prototype/inp/",
                               out_file="test_1.csv", mismatch_file="mm_1.csv",
                               keep_fuzzy=True, keep_higherrank=True)

Mismatches encountered of type {'NONE', 'FUZZY', 'HIGHERRANK', 'EXACT'}
Count NONE: 2
Count FUZZY: 15
Count HIGHERRANK: 5
Count EXACT: 189


In [180]:
tdf1.head()

,usageKey,scientificName,canonicalName,rank,status,confidence,matchType,kingdom,phylum,order,...,classKey,orderKey,familyKey,genusKey,speciesKey,class,acceptedUsageKey,note,synonym,lookupNames
0,3189103.0,Epilobium davuricum Fisch. ex Hornem.,Epilobium davuricum,SPECIES,ACCEPTED,98,EXACT,Plantae,Tracheophyta,Myrtales,...,220.0,690.0,2430.0,3189067.0,3189103.0,Magnoliopsida,3189103,NaN,NaN,Epilobium davuricum
1,2727917.0,Carex appropinquata Schumach.,Carex appropinquata,SPECIES,ACCEPTED,99,EXACT,Plantae,Tracheophyta,Poales,...,196.0,1369.0,7708.0,2721893.0,2727917.0,Liliopsida,2727917,NaN,NaN,Carex appropinquata
2,3152379.0,Malva moschata L.,Malva moschata,SPECIES,ACCEPTED,98,EXACT,Plantae,Tracheophyta,Malvales,...,220.0,941.0,6685.0,3152364.0,3152379.0,Magnoliopsida,3152379,NaN,NaN,Malva moschata
3,5371874.0,Pimpinella major (L.) Huds.,Pimpinella major,SPECIES,ACCEPTED,97,EXACT,Plantae,Tracheophyta,Apiales,...,220.0,1351.0,6720.0,3034784.0,5371874.0,Magnoliopsida,5371874,NaN,NaN,Pimpinella major
4,3039454.0,Frangula alnus Mill.,Frangula alnus,SPECIES,ACCEPTED,99,EXACT,Plantae,Tracheophyta,Rosales,...,220.0,691.0,2407.0,3039453.0,3039454.0,Magnoliopsida,3039454,NaN,NaN,Frangula alnus


In [181]:
mmdf1

,usageKey,scientificName,canonicalName,rank,status,confidence,matchType,kingdom,phylum,order,...,classKey,orderKey,familyKey,genusKey,speciesKey,class,acceptedUsageKey,note,synonym,lookupNames
178,NaN,NaN,NaN,NaN,NaN,100,NONE,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Multiple equal matches for Chara,False,Chara sp
196,NaN,NaN,NaN,NaN,NaN,100,NONE,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Multiple equal matches for Salix alba,False,Salix alba


# Credentials checker

## Verification function

In [182]:
def verify_gbif_credentials(env_path=".env", overwrite=False):
    """
    Ensures GBIF credentials (GBIF_USER, GBIF_MAIL, GBIF_PWD) are stored in environment variables.
    If missing or invalid, prompts the user to enter them and stores them in a .env file for future use.

    Args:
        env_path (str): Path to the .env file.
        overwrite (bool): If True, forces user to re-enter credentials even if they exist.

    Returns:
        dict: A dictionary with keys 'GBIF_USER', 'GBIF_MAIL', 'GBIF_PWD'.
    """
    credentials = ["GBIF_USER", "GBIF_MAIL", "GBIF_PWD"]

    while True:
        # Prompt for credentials if missing or overwrite is True
        for cred in credentials:
            if overwrite or not os.environ.get(cred):
                if cred == "GBIF_PWD":
                    os.environ[cred] = getpass(f"Enter {cred}: ")
                else:
                    os.environ[cred] = input(f"Enter {cred}: ")

        user = os.environ.get("GBIF_USER")
        email = os.environ.get("GBIF_MAIL")
        pwd = os.environ.get("GBIF_PWD")

        # Verify credentials using pygbif
        try:
            result = occurrences.download_list(user=user, pwd=pwd)
            if isinstance(result, dict):
                print("✅ GBIF credentials are valid.")
                break
            else:
                print("⚠️ Unexpected response, please check your credentials.")
                for cred in credentials:
                    os.environ.pop(cred, None)
        except Exception as e:
            print("❌ Invalid credentials or error accessing GBIF API:", e)
            for cred in credentials:
                os.environ.pop(cred, None)

    # Save credentials in .env
    env_file = Path(env_path)
    existing_lines = []
    if env_file.exists():
        existing_lines = env_file.read_text().splitlines()

    for cred in credentials:
        value = os.environ[cred]
        replaced = False
        for i, line in enumerate(existing_lines):
            if line.startswith(f"{cred}="):
                existing_lines[i] = f"{cred}={value}"
                replaced = True
        if not replaced:
            existing_lines.append(f"{cred}={value}")

    env_file.write_text("\n".join(existing_lines) + "\n")
    print(f"Credentials stored in {env_file.resolve()}.")

    return {cred: os.environ[cred] for cred in credentials}

### Example

In [26]:
creds = verify_gbif_credentials()

✅ GBIF credentials are valid.
Credentials stored in C:\Users\niels\Documents\Repositories\BmC\.env.


# Query constructor

## Hardcoded variables

In [193]:
GBIF_COLUMNS = [
    "gbifid", "accessrights", "bibliographiccitation", "language", "license",
    "modified", "publisher", "references", "rightsholder", "type",
    "institutionid", "collectionid", "datasetid", "institutioncode", "collectioncode",
    "datasetname", "ownerinstitutioncode", "basisofrecord", "informationwithheld",
    "datageneralizations", "dynamicproperties", "occurrenceid", "catalognumber",
    "recordnumber", "recordedby", "recordedbyid", "individualcount",
    "organismquantity", "organismquantitytype", "sex", "lifestage",
    "reproductivecondition", "caste", "behavior", "vitality",
    "establishmentmeans", "degreeofestablishment", "pathway",
    "georeferenceverificationstatus", "occurrencestatus", "preparations",
    "disposition", "associatedoccurrences", "associatedreferences",
    "associatedsequences", "associatedtaxa", "othercatalognumbers",
    "occurrenceremarks", "organismid", "organismname", "organismscope",
    "associatedorganisms", "previousidentifications", "organismremarks",
    "materialentityid", "materialentityremarks", "verbatimlabel",
    "materialsampleid", "eventid", "parenteventid", "eventtype",
    "fieldnumber", "eventdate", "eventtime", "startdayofyear", "enddayofyear",
    "year", "month", "day", "verbatimeventdate", "habitat", "samplingprotocol",
    "samplesizevalue", "samplesizeunit", "samplingeffort", "fieldnotes",
    "eventremarks", "locationid", "highergeographyid", "highergeography",
    "continent", "waterbody", "islandgroup", "island", "countrycode",
    "stateprovince", "county", "municipality", "locality", "verbatimlocality",
    "verbatimelevation", "verticaldatum", "verbatimdepth",
    "minimumdistanceabovesurfaceinmeters", "maximumdistanceabovesurfaceinmeters",
    "locationaccordingto", "locationremarks", "decimallatitude",
    "decimallongitude", "coordinateuncertaintyinmeters", "coordinateprecision",
    "pointradiusspatialfit", "verbatimcoordinatesystem", "verbatimsrs",
    "footprintwkt", "footprintsrs", "footprintspatialfit", "georeferencedby",
    "georeferenceddate", "georeferenceprotocol", "georeferencesources",
    "georeferenceremarks", "geologicalcontextid", "earliesteonorlowesteonothem",
    "latesteonorhighesteonothem", "earliesteraorlowesterathem",
    "latesteraorhighesterathem", "earliestperiodorlowestsystem",
    "latestperiodorhighestsystem", "earliestepochorlowestseries",
    "latestepochorhighestseries", "earliestageorloweststage",
    "latestageorhigheststage", "lowestbiostratigraphiczone",
    "highestbiostratigraphiczone", "lithostratigraphicterms", "group",
    "formation", "member", "bed", "identificationid", "verbatimidentification",
    "identificationqualifier", "typestatus", "identifiedby", "identifiedbyid",
    "dateidentified", "identificationreferences", "identificationverificationstatus",
    "identificationremarks", "taxonid", "scientificnameid", "acceptednameusageid",
    "parentnameusageid", "originalnameusageid", "nameaccordingtoid",
    "namepublishedinid", "taxonconceptid", "scientificname",
    "acceptednameusage", "parentnameusage", "originalnameusage",
    "nameaccordingto", "namepublishedin", "namepublishedinyear",
    "higherclassification", "kingdom", "phylum", "class", "order", "superfamily",
    "family", "subfamily", "tribe", "subtribe", "genus", "genericname",
    "subgenus", "infragenericepithet", "specificepithet",
    "infraspecificepithet", "cultivarepithet", "taxonrank",
    "verbatimtaxonrank", "vernacularname", "nomenclaturalcode",
    "taxonomicstatus", "nomenclaturalstatus", "taxonremarks", "datasetKey",
    "publishingcountry", "lastinterpreted", "elevation", "elevationaccuracy",
    "depth", "depthaccuracy", "distancefromcentroidinmeters", "issue",
    "taxonomicissue", "nontaxonomicissue", "mediatype", "hascoordinate",
    "hasgeospatialissues", "taxonKey", "acceptedtaxonKey", "kingdomKey",
    "phylumKey", "classKey", "orderKey", "familyKey", "genusKey",
    "subgenusKey", "taxonKey", "speciesKey", "species", "acceptedscientificname",
    "typifiedname", "protocol", "lastparsed", "lastcrawled", "isinvasive",
    "repatriated", "relativeorganismquantity", "projectid", "issequenced",
    "gbifregion", "publishedbygbifregion", "level0gid", "level0name",
    "level1gid", "level1name", "level2gid", "level2name", "level3gid",
    "level3name", "iucnredlistcategory", "publishingorgKey", "installationKey",
    "institutionKey", "collectionKey", "programmeacronym",
    "hostingorganizationKey", "isincluster", "dwcaextension", "eventdategte",
    "eventdatelte"
]
GBIF_GRIDS = ["EEA", "EQDG", "DMSG", "ISEA3H", "MGRS"]
GBIF_GRID_FUNCTIONS = ["GBIF_EEARGCode", "GBIF_EQDGCode", "GBIF_DMSGCode"]
GBIF_GRID_RESOLUTIONS = [[25, 100, 250, 1000, 10000, 50000,100000],
                    list(range(0,8)),
                    [3600, 1800, 900, 600, 300, 150, 60, 30],
                    list(range(1,23)),
                    [0, 1, 10, 100, 1000, 10000, 100000]]

## Construction function

### wkt string constructor

In [184]:
def bbox2polygon_wkt(bbox):
    """
    Convert a bbox to a wkt style polygon to use within the GBIF SQL query API

    Args
        bbox (tuple<float>): A tuple of floats corresponding to (longitude_min, latitude_min, longitude_max, latitude_max)
    Returns
        polygon.wkt (str): The WKT representation of the bbox in polygon format 
    """
    #Read the tuple as a rectangular geometry
    polygon = shapely.geometry.box(*bbox)
    return polygon.wkt

#### Example

In [8]:
bbox = bbox2polygon_wkt((4.097,50.513,5.086,52.305))
bbox

'POLYGON ((5.086 50.513, 5.086 52.305, 4.097 52.305, 4.097 50.513, 5.086 50.513))'

### Query function

In [197]:
def generate_query(taxonKeys, columns, record_type, wkt_polygon,
                   year_range=None,
                   aggregate=False, include_distinct_observers = True,
                   grid=False, grid_resolution=None, coordinateUncertainty=1000,
                   includeUnknownStatus=True,
                   include_uncertainty=True,
                   issue_flags=["hasCoordinate = TRUE", 
                                "NOT ARRAY_CONTAINS(issue, 'ZERO_COORDINATE')",
                                "NOT ARRAY_CONTAINS(issue, 'COORDINATE_OUT_OF_RANGE')",
                                "NOT ARRAY_CONTAINS(issue, 'COORDINATE_INVALID')",
                                "NOT ARRAY_CONTAINS(issue, 'COUNTRY_COORDINATE_MISMATCH')"]):
    #----VALIDITY CHECK----
    #check if all the selected columns are valid columns that can be selected from the occurrence table
    if not set(columns).issubset(GBIF_COLUMNS):
        raise ValueError(f"The following column(s) ({set(columns)-set(GBIF_COLUMNS)}) are not present in the GBIF data table") 
    #check if the requested record type(s) is(are) valid
    if record_type.lower() not in ["occurrence", "absence", "mixed"]:
        raise ValueError(f"Chosen record type {record_type} is invalid. Please choose either 'occurrence' or 'absence'")
    #Some columns require that we use quotes in order to use due to conflict with reserved Keywords and functions in sql
    reserved_columns = ["group", "order", "type", "references", "class", "language", "year", "month", "day"]
    #Format the column names so they are usable in the query
    quoted_columns = [f'"{col}"' if col in reserved_columns else col for col in columns]

    #----ISSUES AND CONDITIONS----
    #--TIME--
    time_columns = " AND ".join([f'"{col}" IS NOT NULL' for col in columns if col in ["year", "month", "day"]])
    #If a year range is given check if it formatted in a valid way
    if year_range:
        if year_range[0]>year_range[1]:
            raise ValueError(f"Year range invalid (start_year > end_year). Please give a list [start_year, end_year] where (start_year < end_year)")
        year_range_str = f'"year" >= {year_range[0]} AND "year" <= {year_range[1]} AND'
    else:
        year_range_str = ""
    #--ISSUES--
    #Join the issue flags together 
    if issue_flags:
        issue_str = " AND ".join(issue_flags)
    else:
        issue_str = ""
    #--OCCURRENCE STATUS--
    status_map = {"occurrence": ["'PRESENT'"],
                  "absence": ["'ABSENT'"],
                  "mixed": ["'PRESENT'", "'ABSENT'"]}
    status_values = status_map[record_type.lower()].copy()
    if includeUnknownStatus:
        status_values.append("'UNKNOWN'")
    status_clause = "occurrenceStatus IN ({})".format(", ".join(status_values))
    status_str_map = {"occurrence": "occurrences",
                      "absence": "absences",
                      "mixed": "frequency"}
    #----GRID SECTION----
    if grid:
        #Check if grid is valid
        if grid.upper() not in GBIF_GRIDS:
            raise ValueError(f"The specified grid '{grid}' is not a supported grid. Please choose one of the following {GBIF_GRIDS}")
        #Extract the corresponding idx
        grid_idx = GBIF_GRIDS.index(grid.upper())
        #Check if resolution is valid
        if grid_resolution not in GBIF_GRID_RESOLUTIONS[grid_idx]:
            raise ValueError(f"The specified resolution '{grid_resolution}' is not a valid option for the selected grid. Please use one of the following option ({GBIF_GRID_RESOLUTIONS[grid_idx]})")
        #Construct gridding function string 
        gridding_str = f"""{GBIF_GRID_FUNCTIONS[grid_idx]}({grid_resolution}, 
                           decimalLatitude, 
                           decimalLongitude, 
                           COALESCE(coordinateUncertaintyInMeters, {coordinateUncertainty})) AS {grid.lower()}CellCode"""

    #----AGGREGATION----
    #Return aggregated records based on the selected columns, grid cell code and occurrencestatus
    if aggregate:
        #Group based on the columns that where requested, grid cell code and occurrence status
        group_statement = f'GROUP BY {",".join(quoted_columns)}{","+grid.lower()+"CellCode" if grid else ""}, occurrenceStatus'
        aggr_statement = f'COUNT(*) AS {status_str_map[record_type]}'
    distinct_obs_clause = ""
    if include_distinct_observers and aggregate:
        distinct_obs_clause = ", COUNT(DISTINCT recordedBy) as distinctObservers"
    select_uncertainty = include_uncertainty and not aggregate
    uncertainty_string = f", COALESCE(coordinateUncertaintyInMeters, {coordinateUncertainty}) as coordinateUncertaintyInMeters"
    #----SQL QUERY BUILD----
    select_statement = f'''SELECT {",".join(quoted_columns)}
                           {",decimalLatitude, decimalLongitude" if not aggregate else ""}{uncertainty_string if select_uncertainty else ""}
                           {","+aggr_statement if aggregate else ""}
                           {distinct_obs_clause}
                           {"," +gridding_str if grid else ""} FROM occurrence'''
    #Filter conditions for the records
    filter_statement = f"""WHERE GBIF_Within('{wkt_polygon}', decimalLatitude, decimalLongitude) = TRUE
                           AND {time_columns} 
                           AND {year_range_str} {issue_str} 
                           AND taxonKey IN ({','.join(map(str, taxonKeys))}) 
                           AND {status_clause}"""
    return f"{select_statement} {filter_statement} {group_statement if aggregate else ""}"

#### Example

In [188]:
len(list(tdf1["acceptedUsageKey"]))

209

In [198]:
occurrenceGridNA_test = generate_query(list(tdf1["acceptedUsageKey"]), 
                                       ["month", "year", "speciesKey", "countrycode", "gbifid"], 
                                       "occurrence", 
                                       bbox, 
                                       grid="EEA", grid_resolution=100)
print(occurrenceGridNA_test)

SELECT "month","year",speciesKey,countrycode,gbifid
                           ,decimalLatitude, decimalLongitude, COALESCE(coordinateUncertaintyInMeters, 1000) as coordinateUncertaintyInMeters
                           
                           
                           ,GBIF_EEARGCode(100, 
                           decimalLatitude, 
                           decimalLongitude, 
                           COALESCE(coordinateUncertaintyInMeters, 1000)) AS eeaCellCode FROM occurrence WHERE GBIF_Within('POLYGON ((5.086 50.513, 5.086 52.305, 4.097 52.305, 4.097 50.513, 5.086 50.513))', decimalLatitude, decimalLongitude) = TRUE
                           AND "month" IS NOT NULL AND "year" IS NOT NULL 
                           AND  hasCoordinate = TRUE AND NOT ARRAY_CONTAINS(issue, 'ZERO_COORDINATE') AND NOT ARRAY_CONTAINS(issue, 'COORDINATE_OUT_OF_RANGE') AND NOT ARRAY_CONTAINS(issue, 'COORDINATE_INVALID') AND NOT ARRAY_CONTAINS(issue, 'COUNTRY_COORDINATE_MISMATCH') 
             

# Query submitter & downloader

## Submission and downloader

In [199]:
def download_query(
    gbif_query: str,
    target_dir: str = "",
    max_time: int = 3600,
    sleep_time: int = 30
):
    """
    Submit a GBIF SQL download request, poll for completion, and save the ZIP file.
    Automatically creates the target directory if it does not exist.
    """

    # ---- Directory existence check and auto-create --------------------------
    if target_dir:
        try:
            os.makedirs(target_dir, exist_ok=True)
        except Exception as e:
            raise RuntimeError(
                f"Could not create target directory '{target_dir}': {e}"
            )
    else:
        target_dir = "."

    # ---- Credentials --------------------------------------------------------
    creds = verify_gbif_credentials()

    # ---- Submit GBIF download ----------------------------------------------
    try:
        download_key = pygbif.occurrences.download_sql(
            gbif_query,
            user=creds["GBIF_USER"],
            pwd=creds["GBIF_PWD"]
        )
    except Exception as e:
        raise RuntimeError(f"Failed to submit GBIF download: {e}")

    print(f"GBIF download key: {download_key}")

    # ---- Poll download status ----------------------------------------------
    start = time.time()
    while True:
        try:
            metadata = pygbif.occurrences.download_meta(download_key)
        except Exception as e:
            print(f"Warning: metadata fetch failed temporarily: {e}")
            metadata = {"status": "UNKNOWN"}

        status = metadata.get("status", "UNKNOWN")

        if status == "SUCCEEDED":
            print("Download succeeded")
            break
        elif status in ("FAILED", "KILLED"):
            raise RuntimeError(f"GBIF download failed with status: {status}")

        if time.time() - start >= max_time:
            raise TimeoutError(
                f"GBIF download did not finish within {max_time} seconds. "
                f"Last status: {status}"
            )

        print(f"[{int(time.time() - start)}s] Status = {status}. Waiting {sleep_time}s...")
        time.sleep(sleep_time)

    # ---- Download file to target directory ---------------------------------
    try:
        filepath = pygbif.occurrences.download_get(download_key, path=target_dir)
    except Exception as e:
        raise RuntimeError(f"Failed to download GBIF file: {e}")
    return filepath

### Example

In [200]:
path = download_query(occurrenceGridNA_test, "test_download")

✅ GBIF credentials are valid.
Credentials stored in C:\Users\niels\Documents\Repositories\BmC\.env.


INFO:Your sql download key is 0054648-251120083545085


GBIF download key: 0054648-251120083545085
[0s] Status = PREPARING. Waiting 30s...
[30s] Status = RUNNING. Waiting 30s...
[60s] Status = RUNNING. Waiting 30s...
[90s] Status = RUNNING. Waiting 30s...
[120s] Status = RUNNING. Waiting 30s...
[150s] Status = RUNNING. Waiting 30s...
[180s] Status = RUNNING. Waiting 30s...
[210s] Status = RUNNING. Waiting 30s...
[240s] Status = RUNNING. Waiting 30s...
[270s] Status = RUNNING. Waiting 30s...
[300s] Status = RUNNING. Waiting 30s...
[330s] Status = RUNNING. Waiting 30s...
[360s] Status = RUNNING. Waiting 30s...
[390s] Status = RUNNING. Waiting 30s...
[421s] Status = RUNNING. Waiting 30s...
[451s] Status = RUNNING. Waiting 30s...
[481s] Status = RUNNING. Waiting 30s...
[511s] Status = RUNNING. Waiting 30s...
[541s] Status = RUNNING. Waiting 30s...
[571s] Status = RUNNING. Waiting 30s...
[601s] Status = RUNNING. Waiting 30s...
[631s] Status = RUNNING. Waiting 30s...
[661s] Status = RUNNING. Waiting 30s...
[691s] Status = RUNNING. Waiting 30s...


INFO:Download file size: 91282539 bytes


Download succeeded


INFO:On disk at test_download/0054648-251120083545085.zip


# Sparse array construction

In [202]:
ls "test_download/"

 Volume in drive C is OS
 Volume Serial Number is 1C25-B250

 Directory of C:\Users\niels\Documents\Repositories\BmC\test_download

16/12/2025  13:31    <DIR>          .
16/12/2025  15:01    <DIR>          ..
16/12/2025  13:31        91.282.539 0054648-251120083545085.zip
               1 File(s)     91.282.539 bytes
               2 Dir(s)  217.491.705.856 bytes free


In [204]:
import zipfile

zip_path = "test_download/0054648-251120083545085.zip"
extract_to = "test_download/"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_to)

In [205]:
ls "test_download/"

 Volume in drive C is OS
 Volume Serial Number is 1C25-B250

 Directory of C:\Users\niels\Documents\Repositories\BmC\test_download

17/12/2025  09:59    <DIR>          .
17/12/2025  09:54    <DIR>          ..
17/12/2025  09:59       183.209.717 0054648-251120083545085.csv
16/12/2025  13:31        91.282.539 0054648-251120083545085.zip
               2 File(s)    274.492.256 bytes
               2 Dir(s)  216.937.119.744 bytes free


In [207]:
df = pd.read_csv("test_download/0054648-251120083545085.csv", sep="\t")
df.head()

,month,year,specieskey,countrycode,gbifid,decimallatitude,decimallongitude,coordinateuncertaintyinmeters,eeacellcode
0,6,1952,5347644.0,BE,2251588278,50.618214,4.67501,1000.0,100mE39447N30696
1,3,2023,8214667.0,BE,5758699239,50.855000,4.66700,177.0,100mE39457N30962
2,7,2022,2477872.0,BE,5758706298,50.796000,4.69100,177.0,100mE39471N30895
3,10,2022,2477872.0,BE,5758719902,50.783000,4.62700,177.0,100mE39425N30884
4,4,2023,9752617.0,BE,5758735710,50.811000,4.63800,177.0,100mE39433N30916


In [209]:
#df["time"]= pd.to_datetime(df[["year", "month"]].assign(day=1))
df.head()

,month,year,specieskey,countrycode,gbifid,decimallatitude,decimallongitude,coordinateuncertaintyinmeters,eeacellcode,time
0,6,1952,5347644.0,BE,2251588278,50.618214,4.67501,1000.0,100mE39447N30696,1952-06-01
1,3,2023,8214667.0,BE,5758699239,50.855000,4.66700,177.0,100mE39457N30962,2023-03-01
2,7,2022,2477872.0,BE,5758706298,50.796000,4.69100,177.0,100mE39471N30895,2022-07-01
3,10,2022,2477872.0,BE,5758719902,50.783000,4.62700,177.0,100mE39425N30884,2022-10-01
4,4,2023,9752617.0,BE,5758735710,50.811000,4.63800,177.0,100mE39433N30916,2023-04-01


In [212]:
df = df.drop(["countrycode", "gbifid", "time"], axis=1)
df

,month,year,specieskey,decimallatitude,decimallongitude,coordinateuncertaintyinmeters,eeacellcode
0,6,1952,5347644.0,50.618214,4.67501,1000.0,100mE39447N30696
1,3,2023,8214667.0,50.855000,4.66700,177.0,100mE39457N30962
2,7,2022,2477872.0,50.796000,4.69100,177.0,100mE39471N30895
3,10,2022,2477872.0,50.783000,4.62700,177.0,100mE39425N30884
4,4,2023,9752617.0,50.811000,4.63800,177.0,100mE39433N30916
...,...,...,...,...,...,...,...
2738324,5,2016,2926557.0,50.931840,4.39744,2828.0,100mE39290N31067
2738325,5,2016,2926557.0,50.931840,4.39744,2828.0,100mE39278N31043
2738326,5,2016,3128547.0,51.001930,5.02433,2828.0,100mE39729N31122
2738327,5,2016,2926557.0,50.967720,4.51135,2828.0,100mE39379N31098


In [214]:
len(set(df["eeacellcode"]))

884326

In [ ]:
max_lat= 52.305
min_lat= 50.513

max_long=
min_long=
4.097,50.513,5.086,52.305